# MNIST Latent Rectified-Flow DigitDreamer — Analysis

Training now happens in `train.py`:

    uv run python projects/mnist/rectified_flow/train.py

(which itself needs `projects/mnist/autoencoder/train.py` to have run first).
This notebook only loads the resulting AE + DigitDreamer checkpoints for
analysis: loss curves and class-conditional sampling.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import glob

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from chimera.data import MNISTLatentDataModule
from chimera.models import DigitDreamer, DigitDreamerAE
from chimera.modules import RectifiedFlowModule

AE_RUN_DIR = "/mnt/ai/runs/mnist/autoencoder"
RF_RUN_DIR = "/mnt/ai/runs/mnist/rectified_flow"
DATA_DIR = "/mnt/ai/data"
LATENT_CHANNELS = 1
AE_MODEL_VARIANT = "tiny"
MODEL_VARIANT = "tiny"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

## Load the autoencoder

In [ ]:
ae_ckpt = torch.load(
    f"{AE_RUN_DIR}/checkpoints/ae.ckpt", map_location="cpu", weights_only=False
)
ae_state = {
    k.removeprefix("model."): v
    for k, v in ae_ckpt["state_dict"].items()
    if k.startswith("model.")
}
ae = DigitDreamerAE.from_variant(
    AE_MODEL_VARIANT, in_channels=1, latent_dim=LATENT_CHANNELS
)
ae.load_state_dict(ae_state)
ae.to(DEVICE).eval()
print(f"loaded AE ({sum(p.numel() for p in ae.parameters()):,} params)")

## Load the cached latents

Built by `train.py` via `MNISTLatentDataModule`; reused here just for
`latent_mean` / `latent_std` (needed to un-standardize samples) and a
validation batch for the reconstruction sanity check below.

In [ ]:
latent_dm = MNISTLatentDataModule(
    autoencoder=None,
    data_dir=DATA_DIR,
    batch_size=128,
    num_workers=4,
    device=DEVICE,
    image_size=32,
    latent_channels=LATENT_CHANNELS,
)
latent_dm.prepare_data()
latent_dm.setup("fit")
print("cache:", latent_dm.cache_path)

## Load DigitDreamer

In [ ]:
digit_dreamer = DigitDreamer.from_variant(
    MODEL_VARIANT,
    latent_channels=LATENT_CHANNELS,
    latent_size=4,
    patch_size=1,
    n_classes=10,
    n_cond_tokens=4,
    class_dropout_prob=0.1,
)
rf_ckpt = torch.load(
    f"{RF_RUN_DIR}/checkpoints/digit_dreamer.ckpt",
    map_location="cpu",
    weights_only=False,
)
rf_state = {
    k.removeprefix("model."): v
    for k, v in rf_ckpt["state_dict"].items()
    if k.startswith("model.")
}
digit_dreamer.load_state_dict(rf_state)
digit_dreamer.to(DEVICE).eval()
n_params = sum(p.numel() for p in digit_dreamer.parameters())
print(f"loaded DigitDreamer ({n_params / 1e6:.2f}M params)")

## Training curve

In [ ]:
csv_path = sorted(glob.glob(f"{RF_RUN_DIR}/csv/version_*/metrics.csv"))[-1]
m = pd.read_csv(csv_path)
plt.figure(figsize=(6, 4))
for col in ["train/loss_step", "val/loss", "val/loss_epoch"]:
    if col in m.columns:
        d = m.dropna(subset=[col])
        plt.plot(
            d["step"],
            d[col],
            marker=None if col == "train/loss_step" else "o",
            label=col,
        )
plt.xlabel("step")
plt.ylabel("velocity L1")
plt.legend()
plt.title("DigitDreamer rectified-flow loss")
plt.show()

## Sample & visualize

Generate latents for each digit class with classifier-free guidance,
un-standardize, and decode back to images.

In [ ]:
mean = latent_dm.latent_mean.to(DEVICE)
std = latent_dm.latent_std.to(DEVICE)

N_PER_CLASS = 8
CFG_SCALE = 2
labels = torch.arange(10).repeat_interleave(N_PER_CLASS)

# RectifiedFlowModule.sample only needs the model; optimizer/scheduler are
# training-only and unused at inference, so pass None.
sampler = RectifiedFlowModule(digit_dreamer, None, None)
with torch.inference_mode():
    z = sampler.sample(labels, n_steps=50, cfg_scale=CFG_SCALE)
    z = z * std + mean
    imgs = ae.decode(z)

# labels are grouped by class, so nrow=N_PER_CLASS puts one class per row (0-9).
grid = make_grid(imgs.float().cpu().clamp(0, 1), nrow=N_PER_CLASS)
plt.figure(figsize=(N_PER_CLASS, 10))
plt.imshow(to_pil_image(grid), cmap="gray")
plt.title(f"DigitDreamer samples by class (rows 0-9, CFG={CFG_SCALE})")
plt.axis("off")
plt.show()